# Stream A — 01 NER Pipeline

This notebook reads the **processed manifest layer** and extracts named entities for all rows.

It is designed for the current project structure:

```text
data/processed/ooc_pairs/manifests/
cache/stream_a/ner/
```

Python 3.13 note: this notebook tries to use spaCy if available, but has a lightweight regex fallback so it still runs if spaCy is not installed for Python 3.13.


In [1]:
from pathlib import Path
import sys, json, re, pickle, time
from collections import Counter
import pandas as pd
import numpy as np

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root. Run this notebook inside eve-rag.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    MANIFESTS, STREAM_A_CACHE, TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST,
    VERITE_TEST_MANIFEST, MMFAKEBENCH_VAL_MANIFEST, MMFAKEBENCH_TEST_MANIFEST,
    ALL_MANIFEST, FIGURES, TABLES
)

NER_CACHE = STREAM_A_CACHE / "ner"
NER_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("NER_CACHE:", NER_CACHE)


ROOT: E:\eve-rag
NER_CACHE: E:\eve-rag\cache\stream_a\ner


In [2]:
# GPU check. NER usually runs on CPU unless spaCy transformer + GPU is installed.
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        torch.backends.cudnn.benchmark = True
except Exception as e:
    DEVICE = "cpu"
    print("Torch unavailable or failed:", repr(e))

print("NER device policy:", DEVICE)


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
NER device policy: cuda


In [3]:
def read_manifest(path):
    return pd.read_csv(
        path,
        dtype={
            "pair_id": "string",
            "dataset": "string",
            "split": "string",
            "caption": "string",
            "image_path": "string",
            "image_id": "string",
        },
        low_memory=False,
    )

manifests = {
    "newsclippings_train": read_manifest(TRAIN_MANIFEST),
    "newsclippings_val": read_manifest(VAL_MANIFEST),
    "newsclippings_test": read_manifest(TEST_MANIFEST),
    "verite_test": read_manifest(VERITE_TEST_MANIFEST),
    "mmfakebench_val": read_manifest(MMFAKEBENCH_VAL_MANIFEST),
    "mmfakebench_test": read_manifest(MMFAKEBENCH_TEST_MANIFEST),
}

for name, df in manifests.items():
    print(f"{name:<24} rows={len(df):,} empty_captions={(df['caption'].fillna('').str.len()==0).sum():,}")


newsclippings_train      rows=71,072 empty_captions=0
newsclippings_val        rows=7,024 empty_captions=0
newsclippings_test       rows=7,264 empty_captions=0
verite_test              rows=1,001 empty_captions=0
mmfakebench_val          rows=1,000 empty_captions=0
mmfakebench_test         rows=10,000 empty_captions=0


In [4]:
# Try spaCy. Fallback keeps the notebook runnable on Python 3.13.
USE_SPACY = True
SPACY_MODEL = None
nlp = None

try:
    import spacy
    for model_name in ["en_core_web_sm", "en_core_web_trf"]:
        try:
            nlp = spacy.load(model_name)
            SPACY_MODEL = model_name
            break
        except Exception:
            pass
    if nlp is None:
        USE_SPACY = False
        print("spaCy installed but no English model found. Falling back to regex entities.")
    else:
        print("Using spaCy model:", SPACY_MODEL)
except Exception as e:
    USE_SPACY = False
    print("spaCy unavailable; using regex fallback.", repr(e))

ENTITY_LABELS = {"PERSON", "ORG", "GPE", "LOC", "FAC", "NORP", "EVENT", "DATE"}

# Lightweight fallback: extracts capitalized spans and years.
cap_span_re = re.compile(r"\b(?:[A-Z][a-zA-Z]+(?:\s+|$)){1,5}")
year_re = re.compile(r"\b(?:19|20)\d{2}\b")

def regex_entities(text):
    text = "" if pd.isna(text) else str(text)
    ents = []
    seen = set()
    for m in cap_span_re.finditer(text):
        val = m.group(0).strip()
        if len(val) < 3:
            continue
        key = (val, "MISC")
        if key not in seen:
            ents.append({"text": val, "label": "MISC", "start": m.start(), "end": m.end()})
            seen.add(key)
    for m in year_re.finditer(text):
        val = m.group(0)
        key = (val, "DATE")
        if key not in seen:
            ents.append({"text": val, "label": "DATE", "start": m.start(), "end": m.end()})
            seen.add(key)
    return ents

def spacy_entities_batch(texts, batch_size=256):
    out = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        ents = []
        for e in doc.ents:
            if e.label_ in ENTITY_LABELS:
                ents.append({"text": e.text, "label": e.label_, "start": int(e.start_char), "end": int(e.end_char)})
        out.append(ents)
    return out

def extract_entities_for_df(df, batch_size=256):
    texts = df["caption"].fillna("").astype(str).tolist()
    if USE_SPACY and nlp is not None:
        ents = spacy_entities_batch(texts, batch_size=batch_size)
    else:
        ents = [regex_entities(t) for t in texts]
    return dict(zip(df["pair_id"].astype(str), ents))


Using spaCy model: en_core_web_sm


In [5]:
REBUILD_NER = True

ner_outputs = {}
t0 = time.time()

for name, df in manifests.items():
    out_path = NER_CACHE / f"entities_{name}.pkl"
    if out_path.exists() and not REBUILD_NER:
        ents = pickle.load(open(out_path, "rb"))
        print(f"Loaded {name}: {len(ents):,}")
    else:
        ents = extract_entities_for_df(df)
        pickle.dump(ents, open(out_path, "wb"), protocol=4)
        print(f"Built {name}: {len(ents):,}")

    ner_outputs[name] = ents

elapsed = time.time() - t0
print(f"Done in {elapsed/60:.1f} min")


Built newsclippings_train: 71,072
Built newsclippings_val: 7,024
Built newsclippings_test: 7,264
Built verite_test: 1,001
Built mmfakebench_val: 1,000
Built mmfakebench_test: 10,000
Done in 2.3 min


In [6]:
# Merge all NER outputs.
all_ner = {}
for d in ner_outputs.values():
    all_ner.update(d)

pickle.dump(all_ner, open(NER_CACHE / "entities_all.pkl", "wb"), protocol=4)

rows = []
for name, df in manifests.items():
    ents = ner_outputs[name]
    counts = [len(ents.get(pid, [])) for pid in df["pair_id"].astype(str)]
    rows.append({
        "manifest": name,
        "rows": len(df),
        "rows_with_entities": int(sum(c > 0 for c in counts)),
        "mean_entities": float(np.mean(counts)) if counts else 0.0,
        "max_entities": int(max(counts)) if counts else 0,
    })

summary = pd.DataFrame(rows)
summary.to_csv(TABLES / "ner_manifest_summary.csv", index=False)
summary


,manifest,rows,rows_with_entities,mean_entities,max_entities
0,newsclippings_train,71072,70700,3.035584,11
1,newsclippings_val,7024,6994,3.041429,9
2,newsclippings_test,7264,7221,3.033866,10
3,verite_test,1001,957,2.925075,10
4,mmfakebench_val,1000,701,1.613000,7
5,mmfakebench_test,10000,6960,1.631400,10


In [7]:
# Entity type distribution.
type_counter = Counter()
for ents in all_ner.values():
    for e in ents:
        type_counter[e.get("label", "UNK")] += 1

entity_dist = pd.DataFrame(
    [{"entity_type": k, "count": v} for k, v in type_counter.most_common()]
)
entity_dist.to_csv(TABLES / "ner_entity_distribution.csv", index=False)
entity_dist.head(20)


,entity_type,count
0,PERSON,94326
1,GPE,59383
2,ORG,56628
3,DATE,33509
4,NORP,16320
5,FAC,10845
6,LOC,4871
7,EVENT,4119


In [8]:
print("=" * 60)
print("NER complete.")
print("Cache files:")
for p in sorted(NER_CACHE.glob("*.pkl")):
    print(" ", p.relative_to(ROOT), f"{p.stat().st_size/1e6:.2f} MB")
print("Next: visual_grounding_v2_updated.ipynb")
print("=" * 60)


NER complete.
Cache files:
  cache\stream_a\ner\entities_all.pkl 10.99 MB
  cache\stream_a\ner\entities_mmfake.pkl 0.68 MB
  cache\stream_a\ner\entities_mmfakebench_test.pkl 0.83 MB
  cache\stream_a\ner\entities_mmfakebench_val.pkl 0.08 MB
  cache\stream_a\ner\entities_newsclippings_test.pkl 0.85 MB
  cache\stream_a\ner\entities_newsclippings_train.pkl 8.30 MB
  cache\stream_a\ner\entities_newsclippings_val.pkl 0.82 MB
  cache\stream_a\ner\entities_verite.pkl 0.09 MB
  cache\stream_a\ner\entities_verite_test.pkl 0.11 MB
Next: visual_grounding_v2_updated.ipynb
